# Week 3 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-03.md`](../../weeks/week-03.md) · **Given engine:** [`search.py`](search.py)

This notebook is the **teaching walkthrough** of the week-3 studio. It **imports the
reference code from [`solution.py`](solution.py)** — it never re-pastes it — so the
answer shown here is byte-for-byte the one the provided test grades. The correctness
guarantee is separate: `python3 studios/_verify_solutions.py week-03` runs the
unmodified `test_search.py` against `solution.py`.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [1]:
# --- bootstrap: put the week folder (for solution/search) and the repo root
# (for aicourse) on the path, so this runs from anywhere. ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-03"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import solution
from search import EightPuzzle, TinyGraph, Problem, D8, load_instances

## Task 1 — formulate Missionaries & Cannibals (the five components as code)

Formulation is a *design decision* and it is graded. Reference choice: state is
`(m_left, c_left, boat_side)`, initial `(3,3,0)`, goal `(0,0,1)`. We enforce the
"cannibals never outnumber missionaries on either bank" rule inside `actions()` — a
state is only reachable if it is legal — so `result()` stays a pure transition. (Doing
it in `result()` instead is equally defensible; the demo asks students to argue theirs.)
Not covered by a provided test, so we exercise it here: it must solve in the classic
**11** boat trips.

In [2]:
print(inspect.getsource(solution.MissionariesAndCannibals))

p = solution.MissionariesAndCannibals()
node, exp = solution.bfs(p)
assert node is not None, "M&C should be solvable"
assert len(node.path()) == 11, f"classic optimal is 11 trips, got {len(node.path())}"
# UCS (uniform costs here) agrees on length:
node_u, _ = solution.ucs(p)
assert len(node_u.path()) == 11
print(f"M&C solved: {len(node.path())} trips, {exp} expansions")
print("plan:", node.path())

class MissionariesAndCannibals(Problem):
    def __init__(self):
        super().__init__((3, 3, 0), (0, 0, 1))

    def _legal(self, state):
        """No bank may have cannibals outnumbering missionaries when at least one
        missionary stands there. Also enforce the 0..3 population bounds."""
        m_left, c_left, _ = state
        if not (0 <= m_left <= 3 and 0 <= c_left <= 3):
            return False
        m_right, c_right = 3 - m_left, 3 - c_left
        if m_left > 0 and c_left > m_left:
            return False
        if m_right > 0 and c_right > m_right:
            return False
        return True

    def actions(self, state):
        m_left, c_left, boat = state
        legal = []
        for dm, dc in _BOAT_LOADS:
            if boat == 0:                      # boat on left → carry people right
                if dm <= m_left and dc <= c_left:
                    nxt = (m_left - dm, c_left - dc, 1)
                    if self._legal(nxt):
                       

## Task 2 — four algorithms, one search loop

BFS/DFS/UCS are the *same* given `search()` loop with a different frontier discipline
(`fifo` / `lifo` / `priority`) — three one-liners. IDS is the interesting one: it does
**not** call `search()`; it runs a recursive depth-limited DFS and iterates the limit,
buying BFS's completeness at DFS's `O(bd)` memory by re-doing the shallow work.

In [3]:
for fn in (solution.bfs, solution.dfs, solution.ucs,
           solution.depth_limited, solution.ids):
    print(inspect.getsource(fn))

def bfs(problem):
    """Breadth-first search. Complete; optimal only when step costs are uniform."""
    return search(problem, "fifo")

def dfs(problem):
    """Depth-first (graph) search. The explored set in ``search()`` is what makes
    it terminate on cyclic graphs. Not optimal."""
    return search(problem, "lifo")

def ucs(problem):
    """Uniform-cost search. Optimal for any non-negative step costs."""
    return search(problem, "priority")

def depth_limited(problem, limit):
    """Recursive tree search to a fixed depth.

    Returns ``(node, expansions, hit_limit)``: ``node`` is the goal Node or None;
    ``expansions`` counts expanded nodes; ``hit_limit`` is True iff the search was
    cut off by the depth limit (rather than exhausting the subtree). That flag is
    what lets ``ids`` know whether deepening further can still help.

    We skip the immediate parent state to avoid trivial 2-cycles at no memory
    cost (this is *tree* search — no explored set — so the depth bo

### The guarantees that must HOLD

- `test_bfs_optimal_uniform_cost` — on the 8-puzzle every move costs 1, so BFS *is* optimal (len 8).
- `test_ucs_optimal_nonuniform_cost` — UCS returns the true cost-2 path on `TinyGraph`. This is *why UCS exists*.
- `test_dfs_terminates_on_cyclic_graph` — DFS on a 3-cycle terminates only because `search()` keeps an explored set.
- `test_ids_matches_bfs_solution_length` — IDS finds the same length-8 solution as BFS, at `O(bd)` memory.

In [4]:
D8_OPTIMAL_LEN = 8

# BFS optimal under uniform costs
bnode, _ = solution.bfs(EightPuzzle(D8))
assert len(bnode.path()) == D8_OPTIMAL_LEN
print(f"BFS optimal on uniform costs (len={D8_OPTIMAL_LEN})")

# UCS optimal under non-uniform costs
unode, _ = solution.ucs(TinyGraph("start", "A"))
assert unode.g == 2
print("UCS optimal on non-uniform costs (g=2)")

# DFS terminates on a cyclic graph (needs the explored set)
class CyclicGraph(Problem):
    EDGES = {"A": ["B"], "B": ["C"], "C": ["A", "G"], "G": []}
    def actions(self, state): return self.EDGES[state]
    def result(self, state, action): return action
dnode, _ = solution.dfs(CyclicGraph("A", "G"))
assert dnode is not None and dnode.state == "G"
print("DFS terminates on a cyclic graph (explored set does its job)")

# IDS matches BFS length at O(bd) memory
inode, iexp = solution.ids(EightPuzzle(D8))
assert len(inode.path()) == len(bnode.path()) == D8_OPTIMAL_LEN
print(f"IDS matches BFS length ({D8_OPTIMAL_LEN}); ids expansions={iexp:,}")

BFS optimal on uniform costs (len=8)
UCS optimal on non-uniform costs (g=2)
DFS terminates on a cyclic graph (explored set does its job)
IDS matches BFS length (8); ids expansions=233


## The guarantee fails — BFS is suboptimal on non-uniform cost

This is **the point of the exercise** (`test_bfs_suboptimal_nonuniform_cost`). Every
year someone reports it as a bug; it is not. BFS counts *edges*, not *cost*. On
`TinyGraph` the fewest-edges path `start -> A` costs **10**, while the optimal path
`start -> B -> A` costs **2**. BFS's optimality claim has a condition — uniform step
costs — and `TinyGraph` violates it on purpose. A guarantee you cannot watch fail is a
guarantee you have not understood.

In [5]:
bfs_node, _ = solution.bfs(TinyGraph("start", "A"))
ucs_node, _ = solution.ucs(TinyGraph("start", "A"))
print(f"BFS path cost g = {bfs_node.g}  (fewest edges, ignores cost)")
print(f"UCS path cost g = {ucs_node.g}  (true optimum)")
assert bfs_node.g == 10 and ucs_node.g == 2
print("\nGUARANTEE FAIL confirmed: BFS returned the cost-10 path though the optimum is 2 "
      "— optimality holds only under uniform step costs.")

BFS path cost g = 10  (fewest edges, ignores cost)
UCS path cost g = 2  (true optimum)

GUARANTEE FAIL confirmed: BFS returned the cost-10 path though the optimum is 2 — optimality holds only under uniform step costs.


## Task 3 — measurement (expansions vs. depth, the log-scale deliverable)

`solution.measure()` runs all four algorithms across the 8-puzzle bank (depths 4/8/12/16,
ten instances each). `expansions` is the only currency in which these four compare
(scorecard axis 3). Straight lines on a log y-axis mean exponential growth — seeing it
in your own data is the payload of Task 3. We run a subset here to keep the notebook
fast and print mean expansions per (depth, algorithm).

In [6]:
from collections import defaultdict
bank = load_instances()
# subset: 2 instances per depth keeps the headless run quick; measure() does all 10.
small = {d: states[:2] for d, states in bank.items()}
rows = solution.measure(bank=small)

agg = defaultdict(list)
for depth, name, exp, length in rows:
    agg[(depth, name)].append(exp)

print(f"{'depth':>6} {'algo':>5} {'mean_expansions':>16}")
for (depth, name) in sorted(agg):
    vals = agg[(depth, name)]
    print(f"{depth:>6} {name:>5} {sum(vals)/len(vals):>16,.0f}")
# sanity: BFS/UCS/IDS all return the optimal length at each depth
by = {(d, n): L for d, n, e, L in rows}
for d in small:
    assert by[(d, "BFS")] == d, (d, by[(d,'BFS')])
print("\n(BFS solution length matches the verified optimal depth at every depth)")

 depth  algo  mean_expansions
     4   BFS               26
     4   DFS               35
     4   IDS               24
     4   UCS               26
     8   BFS              196
     8   DFS            4,576
     8   IDS              262
     8   UCS              196
    12   BFS            1,708
    12   DFS           15,709
    12   IDS            3,213
    12   UCS            1,708
    16   BFS           11,207
    16   DFS           42,924
    16   IDS           30,019
    16   UCS           11,207

(BFS solution length matches the verified optimal depth at every depth)
